# Export HMC table to LaTeX

In [4]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np


@dataclass(frozen=True)
class CellSpec:
    mean: float | None
    ci_lo: float | None
    ci_hi: float | None
    sddr_bf01: float | None = None


def _finite(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x).reshape(-1)
    return x[np.isfinite(x)]


def _summarize_draws(draws: np.ndarray) -> tuple[float, float, float]:
    d = _finite(draws)
    if d.size == 0:
        raise ValueError("No finite draws.")
    mean = float(np.mean(d))
    lo = float(np.quantile(d, 0.025))
    hi = float(np.quantile(d, 0.975))
    return mean, lo, hi


def _format_num(x: float, *, digits: int = 4) -> str:
    return f"{x:.{digits}f}"


def _format_sddr(bf01: float, *, digits: int = 4) -> str:
    return f"({_format_num(bf01, digits=digits)})"


def _format_cell(cell: CellSpec, *, include_sddr: bool, digits: int = 4) -> str:
    if cell.mean is None:
        return "--"

    lines: list[str] = [_format_num(cell.mean, digits=digits)]
    if include_sddr and cell.sddr_bf01 is not None and np.isfinite(cell.sddr_bf01):
        lines.append(_format_sddr(float(cell.sddr_bf01), digits=digits))

    if len(lines) == 1:
        return lines[0]

    inner = r" \\ ".join(lines)
    return r"\begin{tabular}[c]{@{}c@{}}" + inner + r"\end{tabular}"


def _load_posterior_ds(nc_path: Path):
    import xarray as xr

    for engine in ("h5netcdf", "netcdf4", None):
        try:
            return xr.open_dataset(nc_path, group="posterior", engine=engine)
        except Exception:
            pass
    return xr.open_dataset(nc_path, group="posterior")


def _posterior_draws(posterior_ds, var: str) -> np.ndarray | None:
    if posterior_ds is None or var not in posterior_ds:
        return None
    return np.asarray(posterior_ds[var]).reshape(-1)


def _sddr_bf01_normal(posterior_ds, *, var: str, mu: float, sigma: float) -> float | None:
    try:
        draws = _posterior_draws(posterior_ds, var)
        if draws is None:
            return None
        post = _finite(draws)
        if post.size < 10:
            return None

        n = post.size
        sd = float(np.std(post, ddof=1))
        if not np.isfinite(sd) or sd <= 0:
            return None

        h = 1.06 * sd * (n ** (-1.0 / 5.0))
        if not np.isfinite(h) or h <= 0:
            return None

        z = post / h
        post_at0 = float(np.mean(np.exp(-0.5 * z * z)) / (h * np.sqrt(2.0 * np.pi)))

        prior_at0 = float(
            np.exp(-0.5 * ((0.0 - mu) / sigma) ** 2) / (sigma * np.sqrt(2.0 * np.pi))
        )
        return post_at0 / max(prior_at0, 1e-300)
    except Exception:
        return None


def _cell_for_var(
    posterior_ds,
    *,
    var: str,
    prior_mu_sigma: tuple[float, float] | None = None,
) -> CellSpec:
    draws = _posterior_draws(posterior_ds, var)
    if draws is None:
        return CellSpec(mean=None, ci_lo=None, ci_hi=None, sddr_bf01=None)

    mean, lo, hi = _summarize_draws(draws)
    bf01 = None
    if prior_mu_sigma is not None:
        mu, sigma = prior_mu_sigma
        bf01 = _sddr_bf01_normal(posterior_ds, var=var, mu=mu, sigma=sigma)
    return CellSpec(mean=mean, ci_lo=lo, ci_hi=hi, sddr_bf01=bf01)


def render_ces_table(
    *,
    idata_dir: Path,
    out_path: Path,
    inflation_label: str,
    digits: int = 4,
) -> None:
    inflation_key = inflation_label.upper()
    if inflation_key not in {"CPI", "PPI"}:
        raise ValueError("inflation_label must be 'CPI' or 'PPI'.")

    prior_specs: dict[str, tuple[float, float]] = {
        "alpha": (0.5, 0.2),
        "kappa": (0.1, 0.2),
        "phi_1": (0.7, 0.2),
    }

    model_paths = {
        ("GDPC1 (BN)", "Uncorr."): idata_dir / f"{inflation_key}_CES_orth_output_gap_BN.nc",
        ("GDPC1 (BN)", "Corr."): idata_dir / f"{inflation_key}_CES_output_gap_BN.nc",
        ("Unemployment Gap", "Uncorr."): idata_dir / f"{inflation_key}_CES_orth_unemp_gap.nc",
        ("Unemployment Gap", "Corr."): idata_dir / f"{inflation_key}_CES_unemp_gap.nc",
        ("Inverse of Markup (BN)", "Uncorr."): idata_dir / f"{inflation_key}_CES_orth_markup_BN_inv.nc",
        ("Inverse of Markup (BN)", "Corr."): idata_dir / f"{inflation_key}_CES_markup_BN_inv.nc",
    }

    posterior = {}
    for key, path in model_paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing idata file: {path}")
        posterior[key] = _load_posterior_ds(path)

    coef_vars = ("alpha", "kappa", "phi_1")
    other_vars = ("sigma_e", "sigma_zeta", "rho")

    param_labels = {
        "alpha": r"$\alpha$",
        "kappa": r"$\kappa$",
        "phi_1": r"$\phi_1$",
        "sigma_e": r"$\sigma_e$",
        "sigma_zeta": r"$\sigma_\zeta$",
        "rho": r"Corr$(e_t,\zeta_t)$",
    }

    ordered_cols = list(model_paths.keys())

    def cell(col_key, var: str) -> str:
        include_sddr = var in coef_vars
        prior = prior_specs.get(var) if include_sddr else None
        c = _cell_for_var(posterior[col_key], var=var, prior_mu_sigma=prior)
        return _format_cell(c, include_sddr=include_sddr, digits=digits)

    lines: list[str] = []
    lines.append(r"\begin{tabular}{c|cc|cc|cc}")
    lines.append(r"\toprule")
    lines.append(
        r"Output gap & \multicolumn{2}{c}{GDPC1 (BN)} & \multicolumn{2}{c}{Unemployment Gap} & \multicolumn{2}{c}{Inverse of Markup (BN)} \\"
    )
    lines.append(r"\midrule")
    lines.append(r"Parameter & Uncorr. & Corr. & Uncorr. & Corr. & Uncorr. & Corr. \\")
    lines.append(r"\midrule")

    for var in list(coef_vars) + list(other_vars):
        cells = [cell(col_key, var) for col_key in ordered_cols]
        # Drop parameters that are unused in ALL columns.
        if all(c == "--" for c in cells):
            continue
        row = [param_labels[var], *cells]
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists() or (cand / "output").exists():
            return cand
    return p


if __name__ == "__main__":
    repo_root = find_repo_root(Path.cwd())
    print("repo_root:", repo_root)

    candidates_cpi = [
        repo_root / "code_python" / "output" / "idata" / "hmc_ces_cpi",
        repo_root / "output" / "idata" / "hmc_ces_cpi",
    ]
    idata_dir_cpi = next((p for p in candidates_cpi if p.exists()), None)
    if idata_dir_cpi is None:
        raise FileNotFoundError(
            "Could not find CPI CES idata directory. Tried:\n- "
            + "\n- ".join(map(str, candidates_cpi))
        )

    candidates_ppi = [
        repo_root / "code_python" / "output" / "idata" / "hmc_ces_ppi",
        repo_root / "output" / "idata" / "hmc_ces_ppi",
    ]
    idata_dir_ppi = next((p for p in candidates_ppi if p.exists()), None)
    if idata_dir_ppi is None:
        raise FileNotFoundError(
            "Could not find PPI CES idata directory. Tried:\n- "
            + "\n- ".join(map(str, candidates_ppi))
        )

    out_dir = repo_root / "output" / "tex" / "table" / "hmc_ces"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path_cpi = out_dir / "table_hmc_ces_cpi.tex"
    out_path_ppi = out_dir / "table_hmc_ces_ppi.tex"

    render_ces_table(
        idata_dir=idata_dir_cpi,
        out_path=out_path_cpi,
        inflation_label="CPI",
        digits=4,
    )
    print("written:", out_path_cpi)

    render_ces_table(
        idata_dir=idata_dir_ppi,
        out_path=out_path_ppi,
        inflation_label="PPI",
        digits=4,
    )
    print("written:", out_path_ppi)

repo_root: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_ces/table_hmc_ces_cpi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_ces/table_hmc_ces_ppi.tex


In [5]:
# --- HSA tables and time-varying parameter plots ---

import matplotlib.pyplot as plt


def _safe_vars(ds):
    return [v for v in ds.data_vars if "JitTracer" not in v]


def _flatten_draws(da):
    """Return (n_draws, ...) numpy array by stacking chain/draw if present."""
    if da is None:
        return None
    x = da
    # Common ArviZ dims: chain, draw
    if "chain" in x.dims and "draw" in x.dims:
        x = x.stack(sample=("chain", "draw"))
        x = x.transpose("sample", ...)
        return np.asarray(x)
    return np.asarray(x).reshape(-1, *x.shape[2:]) if x.ndim >= 2 else np.asarray(x).reshape(-1)


def _summarize_path(draws_2d: np.ndarray):
    """Summarize along axis=0 for time series draws shaped (n_draws, T)."""
    d = np.asarray(draws_2d)
    mean = np.nanmean(d, axis=0)
    lo = np.nanquantile(d, 0.025, axis=0)
    hi = np.nanquantile(d, 0.975, axis=0)
    return mean, lo, hi


def _plot_band(x, mean, lo, hi, *, title, ylabel, out_path: Path):
    fig, ax = plt.subplots(1, 1, figsize=(11, 3.2))
    ax.plot(x, mean, lw=2)
    ax.fill_between(x, lo, hi, alpha=0.2)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def _find_idata_dir(effect: str, infl: str) -> Path:
    """Find a directory under results/idata that contains `{infl}_HSA_output_gap_BN.nc` for given effect."""
    effect = effect.lower()
    infl = infl.upper()
    expected = f"{infl}_HSA_output_gap_BN.nc"
    base = repo_root / "output" / "idata"
    hits = []
    for p in base.rglob(expected):
        # Filter by effect keyword in parent path if possible.
        parent = p.parent
        if f"hsa_{effect}" in str(parent).lower():
            hits.append(parent)
    if not hits:
        # fallback: any directory that contains the expected file
        for p in base.rglob(expected):
            hits.append(p.parent)
    if not hits:
        raise FileNotFoundError(f"Could not find any idata dir containing {expected} under {base}")
    # Prefer the shortest path (avoid nested accidental dirs)
    hits = sorted(set(hits), key=lambda x: len(str(x)))
    return hits[0]


def render_hsa_table(*, idata_dir: Path, out_path: Path, infl: str, digits: int = 4) -> None:
    infl = infl.upper()
    prior_specs: dict[str, tuple[float, float]] = {
        "alpha": (0.5, 0.2),
        "kappa": (0.1, 0.2),
        "phi_1": (0.7, 0.2),
        "theta": (0.1, 0.2),
        "rho_1": (0.2, 0.2),
        "rho_2": (0.2, 0.2),
        "kappa_0": (0.1, 0.2),
        "theta_0": (0.1, 0.2),
        "delta": (0.1, 0.2),
        "gamma": (0.0, 0.2),
    }

    model_paths = {
        ("GDPC1 (BN)", "Uncorr."): idata_dir / f"{infl}_HSA_orth_output_gap_BN.nc",
        ("GDPC1 (BN)", "Corr."): idata_dir / f"{infl}_HSA_output_gap_BN.nc",
        ("Unemployment Gap", "Uncorr."): idata_dir / f"{infl}_HSA_orth_unemp_gap.nc",
        ("Unemployment Gap", "Corr."): idata_dir / f"{infl}_HSA_unemp_gap.nc",
        ("Inverse of Markup (BN)", "Uncorr."): idata_dir / f"{infl}_HSA_orth_markup_BN_inv.nc",
        ("Inverse of Markup (BN)", "Corr."): idata_dir / f"{infl}_HSA_markup_BN_inv.nc",
    }

    posterior = {}
    for key, path in model_paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing idata file: {path}")
        posterior[key] = _load_posterior_ds(path)

    # Prefer using constant params if present; otherwise fall back to _0 naming.
    coef_vars = ("alpha", "kappa", "phi_1", "theta", "rho_1", "rho_2")
    other_vars = ("sigma_e", "sigma_zeta", "sigma_u", "sigma_eps")

    param_labels = {
        "alpha": r"$\alpha$",
        "kappa": r"$\kappa$",
        "phi_1": r"$\phi_1$",
        "theta": r"$\theta$",
        "rho_1": r"$\rho_1$",
        "rho_2": r"$\rho_2$",
        "sigma_e": r"$\sigma_e$",
        "sigma_zeta": r"$\sigma_\zeta$",
        "sigma_u": r"$\sigma_u$",
        "sigma_eps": r"$\sigma_\epsilon$",
    }

    ordered_cols = list(model_paths.keys())

    def pick_var(ds, base):
        if base in ds:
            return base
        alt = f"{base}_0"
        return alt if alt in ds else base

    def cell(col_key, var: str) -> str:
        ds = posterior[col_key]
        v = pick_var(ds, var)
        include_sddr = var in coef_vars
        prior = prior_specs.get(var) if include_sddr else None
        c = _cell_for_var(ds, var=v, prior_mu_sigma=prior)
        # CI is not shown (mean + optional SDDR only).
        return _format_cell(c, include_sddr=include_sddr, digits=digits)

    lines: list[str] = []
    lines.append(r"\begin{tabular}{c|cc|cc|cc}")
    lines.append(r"\toprule")
    lines.append(
        r"Output gap & \multicolumn{2}{c}{GDPC1 (BN)} & \multicolumn{2}{c}{Unemployment Gap} & \multicolumn{2}{c}{Inverse of Markup (BN)} \\")
    lines.append(r"\midrule")
    lines.append(r"Parameter & Uncorr. & Corr. & Uncorr. & Corr. & Uncorr. & Corr. \\")
    lines.append(r"\midrule")

    for var in list(coef_vars) + list(other_vars):
        cells = [cell(col_key, var) for col_key in ordered_cols]
        # Drop parameters that are unused in ALL columns.
        if all(c == "--" for c in cells):
            continue
        row = [param_labels[var], *cells]
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


In [6]:
def export_time_varying_plots(*, idata_path: Path, out_dir: Path, title_prefix: str = ""):
    post = _load_posterior_ds(idata_path)
    vars_ = set(_safe_vars(post))

    # kappa_t from kappa_0 + delta * Nbar
    if {"kappa_0", "delta", "Nbar"}.issubset(vars_):
        k0 = _flatten_draws(post["kappa_0"]).reshape(-1, 1)
        d = _flatten_draws(post["delta"]).reshape(-1, 1)
        nbar = _flatten_draws(post["Nbar"])
        # Nbar is typically (draws, T) already
        if nbar.ndim == 1:
            nbar = nbar.reshape(-1, 1)
        kappa_t = k0 + d * nbar
        mean, lo, hi = _summarize_path(kappa_t)
        x = np.arange(mean.shape[0])
        _plot_band(x, mean, lo, hi, title=f"{title_prefix} kappa_t", ylabel=r"$\kappa_t$", out_path=out_dir / "kappa_t.png")

    # theta_t from theta_0 + gamma * Nhat
    if {"theta_0", "gamma", "Nhat"}.issubset(vars_):
        th0 = _flatten_draws(post["theta_0"]).reshape(-1, 1)
        g = _flatten_draws(post["gamma"]).reshape(-1, 1)
        nhat = _flatten_draws(post["Nhat"])
        if nhat.ndim == 1:
            nhat = nhat.reshape(-1, 1)
        theta_t = th0 + g * nhat
        mean, lo, hi = _summarize_path(theta_t)
        x = np.arange(mean.shape[0])
        _plot_band(x, mean, lo, hi, title=f"{title_prefix} theta_t", ylabel=r"$\theta_t$", out_path=out_dir / "theta_t.png")


# Export HSA tables + time-varying plots for (dynamic, steady, full) when available.
for effect in ("dynamic", "steady", "full"):
    for infl in ("CPI", "PPI"):
        try:
            idata_dir = _find_idata_dir(effect, infl)
        except Exception as e:
            print(f"[skip] {effect} {infl}: {e}")
            continue

        out_dir = repo_root / "output" / "tex" / "table" / f"hmc_hsa_{effect}"
        out_path = out_dir / f"table_hmc_hsa_{effect}_{infl.lower()}.tex"
        try:
            render_hsa_table(idata_dir=idata_dir, out_path=out_path, infl=infl, digits=4)
            print("written:", out_path)
        except Exception as e:
            print(f"[error] table {effect} {infl}: {e}")

        # time-varying plots: generate per gap for both correlated (HSA_*) and orth/uncorrelated (HSA_orth_*).
        fig_base = repo_root / "output" / "fig" / f"hmc_hsa_{effect}" / infl.lower()
        for gap in ("output_gap_BN", "unemp_gap", "markup_BN_inv"):
            variants = [
                ("corr", idata_dir / f"{infl}_HSA_{gap}.nc"),
                ("uncorr", idata_dir / f"{infl}_HSA_orth_{gap}.nc"),
            ]
            for corr_key, idata_path in variants:
                if not idata_path.exists():
                    continue
                outp = fig_base / gap / corr_key
                export_time_varying_plots(
                    idata_path=idata_path,
                    out_dir=outp,
                    title_prefix=f"{infl} {effect} {gap} {corr_key}",
                )


written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_dynamic/table_hmc_hsa_dynamic_cpi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_dynamic/table_hmc_hsa_dynamic_ppi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_steady/table_hmc_hsa_steady_cpi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_steady/table_hmc_hsa_steady_ppi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_full/table_hmc_hsa_full_cpi.tex
written: /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/hmc_hsa_full/table_hmc_hsa_full_ppi.tex
